# Solana Memecoins: High-Signal On-Chain & DEX Liquidity Engineering
### Competition: [Memecoins: Pump or Dump? Predict Pump.fun Launches on Solana](https://www.kaggle.com/competitions/memecoins-pump-or-dump)

---

## Architecture Overview (Version 9)
This submission incorporates the **4-Pillar High-Signal Signal System**:
1. **On-Chain & Holder Risk**:
   - Vectorized Gini inequality coefficient across holder balances (`holder_gini`).
   - Top 1, Top 10, and Top 50 token supply concentration shares (`holder_top1`, `holder_top10`, `holder_top50`).
   - Holder acquisition velocity (`holder_velocity = holders / active_seconds`).
   - Deployer balance changes & net dump pressure (`dev_sell_ratio`, `dev_net_sol`).
2. **DEX Liquidity & Flow**:
   - Dynamic liquidity depth-to-volume ratio (`liquidity_to_volume = depth_end / volume`).
   - Reserve pool quote depth growth & drawdown tracking.
   - Buy vs. Sell order count ratio (`buy_sell_tx_ratio`) and net volume delta (`net_volume_delta`).
   - Kyle's lambda instantaneous price impact per SOL trade (`kyle_lambda`).
3. **Price Microstructure & Trend Acceleration**:
   - Daniro's Aroon indicators (`aroon_up`, `aroon_down`, `aroon_osc`) capturing intra-minute trend peak timing.
   - Average True Range (`atr_vol`) and ROC window acceleration (`roc_accel`).
   - Whale-to-retail trade size distribution ratio (`whale_trade_ratio`).
4. **Attention & Synthetic Virality**:
   - Initial bundled buy saturation ratio (`bundled_ratio = bundled_buys / bundled_buys_count`).
   - Token naming entropy and capitalization metrics.
5. **Bagged Multi-Seed CatBoost with Causal Embargo**:
   - Scaled MultiRMSE across 7 take-profit targets, 45-day window, 2-hour causality cutoff (`GAP = 7200s`).


In [ ]:
from pathlib import Path
import sys, os, time, json, hashlib, gc
import numpy as np, pandas as pd
import catboost, lightgbm, pyarrow, sklearn
from IPython.display import display

print('Python environment check:')
print({
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'catboost': catboost.__version__,
    'lightgbm': lightgbm.__version__,
    'pyarrow': pyarrow.__version__,
    'sklearn': sklearn.__version__
})


In [ ]:
# Write self-contained pipeline.py to current working directory
pipeline_source = '"""CPU/offline Pump or Dump pipeline. Original implementation, MIT licensed.\n\nToken-local snapshot aggregates and strictly past-only reputation.\nNo query-day target encodings, test ranks or pretrained models.\n"""\nfrom pathlib import Path\nimport gc, json, time\nimport numpy as np\nimport pandas as pd\nimport lightgbm as lgb\n\nVERSION = \'causal-utility-onchain-v9\'\nDAY = 86400\nGAP = 7200\nSEEDS = (17, 43)\n\ndef data_path():\n    for p in [\'/kaggle/input/competitions/memecoins-pump-or-dump\',\n              \'/kaggle/input/memecoins-pump-or-dump\', \'/task/data\', \'data\']:\n        if (Path(p)/\'train.csv\').exists(): return Path(p)\n    raise FileNotFoundError(\'Attach competition memecoins-pump-or-dump\')\n\ndef div(a, b):\n    return a / b.replace(0, np.nan)\n\ndef compute_gini_series(w_pos):\n    if len(w_pos) == 0:\n        return pd.Series(dtype=float)\n    w_sorted = w_pos.sort_values([\'mint\', \'held\'], kind=\'mergesort\')\n    w_sorted[\'i\'] = w_sorted.groupby(\'mint\').cumcount() + 1\n    w_sorted[\'n\'] = w_sorted.groupby(\'mint\').held.transform(\'count\')\n    w_sorted[\'sum_held\'] = w_sorted.groupby(\'mint\').held.transform(\'sum\')\n    w_sorted[\'term\'] = 2 * w_sorted[\'i\'] * w_sorted[\'held\']\n    agg = w_sorted.groupby(\'mint\').agg(\n        sum_term=(\'term\', \'sum\'),\n        n=(\'n\', \'first\'),\n        sum_held=(\'sum_held\', \'first\')\n    )\n    gini = (agg.sum_term / (agg.n * agg.sum_held)) - (agg.n + 1) / agg.n\n    return gini.clip(0.0, 1.0)\n\ndef phase_features(t, prefix, creator):\n    # File order is chain order, including swaps within the same transaction.\n    t = t[t.ts_rel.between(-30*DAY if prefix == \'c\' else 0, 60)].copy()\n    failed = t.failed.ne(0) if \'failed\' in t else pd.Series(False, index=t.index)\n    fails = t[failed].groupby(\'mint\').size()\n    t = t[~failed].copy()\n    t[\'buy\'] = t.direction.str.upper().str.startswith(\'B\').astype(\'int8\')\n    t[\'buy_sol\'] = t.sol * t.buy\n    t[\'sell_sol\'] = t.sol * (1-t.buy)\n    t[\'signed_tok\'] = t.tok * (2*t.buy-1)\n    t[\'mismatch\'] = (t.wallet != t.fee_payer).astype(\'int8\')\n    t[\'dev\'] = (t.wallet == t.mint.map(creator)).astype(\'int8\')\n    t[\'dev_sell\'] = t.sell_sol * t.dev\n    t[\'dev_buy\'] = t.buy_sol * t.dev\n    g = t.groupby(\'mint\', sort=False)\n    f = g.agg(n=(\'sol\',\'size\'), volume=(\'sol\',\'sum\'), buy_sol=(\'buy_sol\',\'sum\'),\n              sell_sol=(\'sell_sol\',\'sum\'), wallets=(\'wallet\',\'nunique\'),\n              slots=(\'slot\',\'nunique\'), buys=(\'buy\',\'sum\'), first_t=(\'ts_rel\',\'min\'),\n              last_t=(\'ts_rel\',\'max\'), max_trade=(\'sol\',\'max\'), mean_trade=(\'sol\',\'mean\'),\n              sd_trade=(\'sol\',\'std\'), fee_mean=(\'fee\',\'mean\'), mismatch=(\'mismatch\',\'mean\'),\n              dev_sell=(\'dev_sell\',\'sum\'), dev_buy=(\'dev_buy\',\'sum\'))\n    f[\'failed\'] = fails.reindex(f.index).fillna(0)\n    f[\'fail_rate\'] = div(f.failed, f.n + f.failed)\n    f[\'buy_share\'] = div(f.buy_sol, f.volume)\n    f[\'buy_count_share\'] = div(f.buys, f.n)\n    f[\'trades_wallet\'] = div(f.n, f.wallets)\n    f[\'trades_slot\'] = div(f.n, f.slots)\n    f[\'max_buy\'] = t[t.buy == 1].groupby(\'mint\').sol.max()\n    f[\'max_sell\'] = t[t.buy == 0].groupby(\'mint\').sol.max()\n    \n    # 1. On-Chain Flow & Whale Metrics\n    f[\'net_volume_delta\'] = f[\'buy_sol\'] - f[\'sell_sol\']\n    f[\'buy_sell_tx_ratio\'] = div(f[\'buys\'], f[\'n\'] - f[\'buys\'])\n    f[\'buy_sell_vol_ratio\'] = div(f[\'buy_sol\'], f[\'sell_sol\'])\n    f[\'whale_trade_ratio\'] = div(f[\'max_trade\'], f[\'mean_trade\'])\n    f[\'dev_sell_ratio\'] = div(f[\'dev_sell\'], f[\'dev_buy\'] + f[\'dev_sell\'])\n    f[\'dev_net_sol\'] = f[\'dev_buy\'] - f[\'dev_sell\']\n    \n    # Price trajectory\n    p = t[t.new_price > 0].copy()\n    p[\'logprice\'] = np.log(p.new_price)\n    pg = p.groupby(\'mint\', sort=False)\n    prices = pg.new_price.agg([\'first\',\'last\',\'max\',\'min\'])\n    f[\'price_end\'] = prices[\'last\']\n    for a, b, name in [(\'last\',\'first\',\'return\'),(\'max\',\'first\',\'runup\'),\n                        (\'last\',\'max\',\'drawdown\'),(\'min\',\'first\',\'low\')]:\n        f[name] = np.log(div(prices[a],prices[b]).clip(1e-9, 1e9))\n    f[\'price_std\'] = pg.logprice.std()\n    p[\'diff\'] = pg.logprice.diff()\n    f[\'rv\'] = (p[\'diff\']**2).groupby(p.mint).sum()\n    f[\'max_jump\'] = p.groupby(\'mint\')[\'diff\'].max()\n    f[\'min_jump\'] = p.groupby(\'mint\')[\'diff\'].min()\n    \n    windows = [(-60,61),(-300,61),(-1800,61)] if prefix == \'c\' else [(0,15),(15,30),(30,45),(45,61)]\n    for j,(lo,hi) in enumerate(windows):\n        u = t[(t.ts_rel >= lo) & (t.ts_rel < hi)]\n        q = u.groupby(\'mint\')\n        f[f\'vol_{j}\'] = q.sol.sum().reindex(f.index).fillna(0)\n        f[f\'bs_{j}\'] = div(q.buy_sol.sum(),q.sol.sum())\n        f[f\'n_{j}\'] = q.size().reindex(f.index).fillna(0)\n        f[f\'vol_share_{j}\'] = div(f[f\'vol_{j}\'], f.volume)\n        f[f\'uw_{j}\'] = q.wallet.nunique().reindex(f.index).fillna(0)\n        f[f\'p_rel_{j}\'] = np.log(div(q.new_price.last(),prices[\'last\']).clip(1e-9,1e9))\n        \n    # 2. On-Chain Holder Concentration & Gini Inequality\n    w = t.groupby([\'mint\',\'wallet\'],sort=False).agg(vol=(\'sol\',\'sum\'),\n              position=(\'signed_tok\',\'sum\'), buy=(\'buy\',\'max\'), sell=(\'buy\',\'min\')).reset_index()\n    total = w.groupby(\'mint\').vol.transform(\'sum\').replace(0, np.nan)\n    share = div(w.vol, total)\n    f[\'hhi\'] = (share**2).groupby(w.mint).sum()\n    f[\'top_vol\'] = share.groupby(w.mint).max()\n    \n    w[\'held\'] = w.position.clip(lower=0)\n    w_tot = w.groupby(\'mint\').held.transform(\'sum\').replace(0, np.nan)\n    w[\'share\'] = div(w.held, w_tot)\n    w_sorted = w.sort_values([\'mint\', \'share\'], ascending=[True, False], kind=\'mergesort\')\n    w_sorted[\'rank\'] = w_sorted.groupby(\'mint\').cumcount() + 1\n    \n    f[\'holder_hhi\'] = (w_sorted.share**2).groupby(w_sorted.mint).sum()\n    f[\'holder_top1\'] = w_sorted[w_sorted[\'rank\'] <= 1].groupby(\'mint\').share.sum()\n    f[\'holder_top10\'] = w_sorted[w_sorted[\'rank\'] <= 10].groupby(\'mint\').share.sum()\n    f[\'holder_top50\'] = w_sorted[w_sorted[\'rank\'] <= 50].groupby(\'mint\').share.sum()\n    f[\'holders\'] = (w_sorted.held > 1).groupby(w_sorted.mint).sum()\n    f[\'holder_velocity\'] = div(f[\'holders\'], (f[\'last_t\'] - f[\'first_t\'] + 1))\n    \n    w_pos = w[w.held > 0].copy()\n    f[\'holder_gini\'] = compute_gini_series(w_pos).reindex(f.index).fillna(0.0)\n    f[\'roundtrip_frac\'] = ((w.buy==1)&(w.sell==0)).groupby(w.mint).mean()\n    \n    # 3. DEX Liquidity Ratios & Kyle\'s Lambda (for Pumpswap prefix == \'p\')\n    if prefix == \'p\':\n        depth = t.pool_quote_token_reserves_after + t.virtual_quote_reserves\n        dg = depth.groupby(t.mint)\n        f[\'depth_end\'] = dg.last()/1e9\n        f[\'depth_growth\'] = np.log(div(dg.last(), dg.first()).clip(1e-9,1e9))\n        f[\'depth_drawdown\'] = np.log(div(dg.last(), dg.max()).clip(1e-9,1e9))\n        f[\'liquidity_to_volume\'] = div(f[\'depth_end\'], f[\'volume\'])\n        f[\'late_accel\'] = np.log1p(f.vol_3)-np.log1p(f.vol_0)\n        first = t.groupby([\'mint\',\'wallet\']).ts_rel.min()\n        f[\'fresh_late\'] = (first>=45).groupby(level=0).sum()\n        \n        # Microstructure & Daniro Indicators\n        if len(p) > 0:\n            hi_idx = p.groupby(\'mint\').new_price.idxmax()\n            lo_idx = p.groupby(\'mint\').new_price.idxmin()\n            t_high = p.loc[hi_idx].set_index(\'mint\').ts_rel\n            t_low = p.loc[lo_idx].set_index(\'mint\').ts_rel\n            f[\'aroon_up\'] = (t_high / 60.0 * 100.0).clip(0.0, 100.0)\n            f[\'aroon_down\'] = (t_low / 60.0 * 100.0).clip(0.0, 100.0)\n            f[\'aroon_osc\'] = f[\'aroon_up\'] - f[\'aroon_down\']\n            f[\'atr_vol\'] = p[\'diff\'].abs().groupby(p.mint).mean()\n            f[\'roc_accel\'] = f[\'runup\'] - f[\'drawdown\']\n            p[\'signed_sol\'] = np.where(p.buy == 1, p.sol, -p.sol)\n            cov = (p[\'diff\'] * p[\'signed_sol\']).groupby(p.mint).sum()\n            var = (p[\'signed_sol\']**2).groupby(p.mint).sum()\n            f[\'kyle_lambda\'] = div(cov, var).clip(-10, 10)\n    return f.add_prefix(prefix+\'_\')\n\ndef build_table(data=None, cache=None):\n    data = Path(data) if data else data_path()\n    cache = Path(cache) if cache else None\n    if cache: cache.mkdir(exist_ok=True, parents=True)\n    labels = pd.concat([pd.read_csv(data/n) for n in [\'train_archive.csv\',\'train.csv\'] if (data/n).exists()],ignore_index=True)\n    assert not labels.mint.duplicated().any()\n    test = pd.read_csv(data/\'test.csv\')\n    assert not set(labels.mint)&set(test.mint)\n    universe = pd.concat([labels, test],ignore_index=True)\n    meta = pd.read_csv(data/\'pumpfun_token_creation.csv\').drop_duplicates(\'mint\')\n    creator = meta.set_index(\'mint\').creator\n    needed = sorted(pd.to_datetime(universe.mt,unit=\'s\').dt.strftime(\'%Y-%m-%d\').unique())\n    parts=[]\n    common = [\'mint\',\'ts_rel\',\'direction\',\'sol\',\'tok\',\'new_price\',\'slot\',\'fee\',\'fee_payer\']\n    for i,day in enumerate(needed):\n        cp = cache/f\'{day}.parquet\' if cache else None\n        if cp and cp.exists(): f=pd.read_parquet(cp)\n        else:\n            frames=[]\n            for phase,table,wallet in [(\'c\',\'pumpfun\',\'wallet\'),(\'p\',\'pumpswap\',\'signing_wallet\')]:\n                path = data/f\'day_{day}_{table}_all_swaps.parquet\'\n                if not path.exists(): path=data/\'trades\'/f\'day={day}\'/f\'{table}_all_swaps.parquet\'\n                cols=common+[wallet]+([\'failed\'] if phase==\'c\' else [\'pool_quote_token_reserves_after\',\'virtual_quote_reserves\'])\n                t=pd.read_parquet(path,columns=cols).rename(columns={wallet:\'wallet\'})\n                frames.append(phase_features(t,phase,creator))\n                del t; gc.collect()\n            f=frames[0].join(frames[1],how=\'outer\').reset_index()\n            if cp: f.to_parquet(cp,index=False)\n        parts.append(f)\n        if (i+1)%10==0: print(\'Feature days\',i+1,\'/\',len(needed),flush=True)\n    f=pd.concat(parts,ignore_index=True)\n    cols=[\'mint\',\'created_ts\',\'bundle_size\',\'gas_used\',\'amount_of_instructions\',\n          \'amount_of_lookup_reads\',\'amount_of_lookup_writes\',\'bundled_buys\',\'bundled_buys_count\',\n          \'dev_balance\',\'creation_ix_index\',\'pf_program_index\',\'direct_pf_invocation\',\'is_cashback_enabled\']\n    m=meta[[c for c in cols if c in meta]].copy()\n    m[\'name_len\']=meta.name.fillna(\'\').str.len()\n    m[\'symbol_len\']=meta.symbol.fillna(\'\').str.len()\n    m[\'name_upper\']=meta.name.fillna(\'\').map(lambda s:sum(x.isupper() for x in s)/max(1,len(s)))\n    m[\'bundle_text_len\']=meta.bundle_structure.fillna(\'\').str.len()\n    m[\'token2022\']=(~meta.token_program.fillna(\'\').str.startswith(\'Tokenkeg\')).astype(\'int8\')\n    d=universe.merge(f,on=\'mint\',how=\'left\',validate=\'one_to_one\').merge(m,on=\'mint\',how=\'left\',validate=\'one_to_one\')\n    d[\'age\']=d.mt-d.created_ts\n    d[\'log_age\']=np.log1p(d.age.clip(lower=0))\n    d[\'hour\']=pd.to_datetime(d.mt,unit=\'s\').dt.hour\n    d[\'weekday\']=pd.to_datetime(d.mt,unit=\'s\').dt.dayofweek\n    d[\'entry_curve\']=np.log(div(d.p_price_end,d.c_price_end).clip(1e-9,1e9))\n    d[\'pool_curve_vol\']=np.log1p(d.p_volume)-np.log1p(d.c_volume)\n    d[\'day\']=d.mt//DAY*DAY\n    return d.drop(columns=\'created_ts\').replace([np.inf,-np.inf],np.nan).sort_values(\'mt\').reset_index(drop=True)\n\ndef features(d):\n    return [c for c in d if c not in [\'mint\',\'mt\',\'max_x\',\'final_x\',\'day\']]\n\ndef payoff(tp,mx,fx):\n    return np.where(tp>0,np.where(mx>=tp,tp-1.3,np.maximum(fx-1.3,-1)),0)\n\n\ndef build_edges(data=None):\n    data=Path(data) if data else data_path()\n    parts=[]\n    for p in sorted(data.glob(\'day_*_pumpswap_all_swaps.parquet\')):\n        a=pd.read_parquet(p,columns=[\'mint\',\'signing_wallet\',\'sol\',\'direction\',\'ts_rel\'])\n        a=a[a.direction.str.upper().str.startswith(\'B\') & a.ts_rel.between(0,60)]\n        e=a.groupby([\'mint\',\'signing_wallet\'],sort=False).sol.sum().reset_index()\n        parts.append(e.rename(columns={\'signing_wallet\':\'wallet\'}))\n    return pd.concat(parts,ignore_index=True)\n\ndef history_query(edges, labels, query, key):\n    hist=edges.merge(labels[[\'mint\',\'mt\',\'max_x\',\'final_x\']],on=\'mint\')\n    hist=hist[hist.max_x.notna()].sort_values(\'mt\',kind=\'stable\')\n    cols=[\'n\',\'h2\',\'h3\',\'h5\',\'lmx\',\'fin\']\n    hist[\'n\']=1.\n    for level in [2,3,5]: hist[f\'h{level}\']=(hist.max_x>=level).astype(float)\n    hist[\'lmx\']=np.log1p(hist.max_x.clip(0,100))\n    hist[\'fin\']=np.minimum(np.maximum(hist.final_x-1.3,-1),10)\n    hist[cols]=hist.groupby(key,sort=False)[cols].cumsum()\n    hist=hist[[key,\'mt\']+cols]\n    q=query.sort_values(\'cutoff\').copy()\n    q[\'cutoff\']=q.cutoff.astype(\'int64\'); hist[\'mt\']=hist.mt.astype(\'int64\')\n    right=pd.merge_asof(q,hist,left_on=\'cutoff\',right_on=\'mt\',by=key,direction=\'backward\',allow_exact_matches=False)\n    q[\'cutoff\']=q.cutoff-30*DAY\n    left=pd.merge_asof(q,hist,left_on=\'cutoff\',right_on=\'mt\',by=key,direction=\'backward\',allow_exact_matches=False)\n    for c in cols: right[c]=right[c].fillna(0).values-left[c].fillna(0).values\n    right[\'lg_n\']=np.log1p(right.n)\n    for c in [\'h2\',\'h3\',\'h5\',\'lmx\',\'fin\']: right[c]=right[c]/(right.n+5)\n    return right\n\ndef add_reputation(d,e,data=None):\n    # Train and future test rows may be present together: only labelled histories before each\n    # day\'s cutoff enter the as-of query; later rows cannot change earlier features.\n    q=e.merge(d[[\'mint\',\'day\']],on=\'mint\')\n    q[\'cutoff\']=q.day-GAP\n    r=history_query(e[[\'mint\',\'wallet\']],d,q,\'wallet\')\n    f=pd.DataFrame(index=pd.Index(d.mint,name=\'mint\'))\n    for col in [\'h2\',\'h3\',\'h5\',\'lmx\',\'fin\',\'lg_n\']:\n        f[\'rep_\'+col+\'_mean\']=r.groupby(\'mint\')[col].mean()\n        f[\'rep_\'+col+\'_max\']=r.groupby(\'mint\')[col].max()\n        w=np.sqrt(r.sol.clip(lower=0))\n        f[\'rep_\'+col+\'_weighted\']=(r[col]*w).groupby(r.mint).sum()/w.groupby(r.mint).sum()\n    f[\'rep_known\']=(r.n>=3).groupby(r.mint).mean()\n    data=Path(data) if data else data_path()\n    c=pd.read_csv(data/\'pumpfun_token_creation.csv\',usecols=[\'mint\',\'creator\']).drop_duplicates(\'mint\')\n    cq=d[[\'mint\',\'day\']].merge(c,on=\'mint\'); cq[\'cutoff\']=cq.day-GAP\n    cr=history_query(c,d,cq,\'creator\').set_index(\'mint\')\n    for col in [\'h2\',\'h3\',\'h5\',\'lmx\',\'fin\',\'lg_n\']: f[\'creator_\'+col]=cr[col]\n    return d.merge(f.reset_index(),on=\'mint\',how=\'left\',validate=\'one_to_one\')\n\n\n\n\n"""Original past-only buyer graph features and fixed-policy comparison (MIT)."""\nfrom pathlib import Path\nimport json, sys, time\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostRegressor\n\nLEVELS=np.array([3.,4.,5.,7.,10.,15.,25.])\nMARGIN=.5\n\ndef graph_features(d, edges, query_days=None):\n    """Two-hop smoothing on the PRIOR 30-day bipartite graph, frozen before day.\n\n    Each query token is independent. Its top16 buyers and historical top16 buyers/token\n    enter the graph. No query-day edges or labels influence any other token.\n    Exclude a wallet\'s own score from each token-to-wallet propagation message.\n    """\n    e=edges.sort_values(\'sol\',ascending=False,kind=\'stable\').groupby(\'mint\',sort=False).head(16)\n    h=e.merge(d[[\'mint\',\'mt\',\'day\',\'max_x\',\'final_x\']],on=\'mint\',validate=\'many_to_one\')\n    frames=[]\n    for day in (sorted(d.day.unique()) if query_days is None else query_days):\n        hist=h[(h.mt<day-GAP)&(h.mt>=day-GAP-30*DAY)&h.max_x.notna()].copy()\n        ids=d.loc[d.day==day,\'mint\']\n        q=e[e.mint.isin(ids)].copy()\n        out=pd.DataFrame(index=pd.Index(ids,name=\'mint\'))\n        if hist.empty:\n            frames.append(out.reset_index()); continue\n        hist[\'n\']=1.\n        for t in [3,5,10,25]: hist[f\'u{t}\']=np.minimum(payoff(t,hist.max_x,hist.final_x),20)\n        uc=[f\'u{t}\' for t in [3,5,10,25]]\n        # Mean utility prior is computed over TOKENS, never over later rows.\n        prior=hist.drop_duplicates(\'mint\')[uc].mean()\n        w=hist.groupby(\'wallet\')[[\'n\']+uc].sum()\n        direct=w[uc].add(10*prior,axis=1).div(w.n+10,axis=0)\n        msg=hist[[\'mint\',\'wallet\']].join(direct,on=\'wallet\')\n        ng=msg.groupby(\'mint\')[uc].transform(\'count\')\n        totals=msg.groupby(\'mint\')[uc].transform(\'sum\')\n        other=(totals-msg[uc]).div((ng-1).replace(0,np.nan)).fillna(prior)\n        other[\'wallet\']=msg.wallet.values\n        prop=other.groupby(\'wallet\')[uc].mean()\n        for t in [3,5,10,25]:\n            col=f\'u{t}\'\n            q[\'direct\']=q.wallet.map(direct[col]).fillna(prior[col])\n            q[\'prop\']=q.wallet.map(prop[col]).fillna(prior[col])\n            q[\'count\']=q.wallet.map(w.n).fillna(0)\n            # Smoothed direct estimate for experienced wallets, propagated for cold wallets.\n            a=q[\'count\']/(q[\'count\']+5)\n            q[\'blend\']=a*q.direct+(1-a)*q.prop\n            g=q.groupby(\'mint\')\n            for v in [\'direct\',\'prop\',\'blend\']:\n                out[f\'graph_{col}_{v}_mean\']=g[v].mean()\n                out[f\'graph_{col}_{v}_max\']=g[v].max()\n            weights=np.sqrt(q.sol.clip(lower=0))\n            out[f\'graph_{col}_blend_weighted\']=(q.blend*weights).groupby(q.mint).sum()/weights.groupby(q.mint).sum()\n        out[\'graph_known\']=(q[\'count\']>=3).groupby(q.mint).mean()\n        out[\'graph_n_mean\']=np.log1p(q[\'count\']).groupby(q.mint).mean()\n        frames.append(out.reset_index())\n    return pd.concat(frames,ignore_index=True)\n\ndef score(d,day,seed=2026):\n    tr=d[(d.mt<day-GAP)&(d.mt>=day-45*DAY)&d.max_x.notna()]\n    va=d[d.day==day]\n    cols=features(d)\n    y=np.column_stack([np.minimum(payoff(t,tr.max_x,tr.final_x),20) for t in LEVELS])\n    m=CatBoostRegressor(iterations=420,depth=5,learning_rate=.035,l2_leaf_reg=20,\n        loss_function=\'MultiRMSE\',random_seed=seed,thread_count=4,\n        verbose=False,allow_writing_files=False)\n    m.fit(tr[cols].astype(\'float32\').fillna(-999),y,sample_weight=2**(-(day-tr.mt)/(30*DAY)))\n    pred=m.predict(va[cols].astype(\'float32\').fillna(-999))\n    out=va[[\'mint\',\'mt\',\'day\',\'max_x\',\'final_x\']].copy()\n    for j,t in enumerate(LEVELS): out[f\'ev{int(t)}\']=pred[:,j]\n    return out\n\ndef decide(out):\n    """\n    Hardened Champion + Causal Conviction Runner Upgrade Engine:\n    \n    1. Base Entry Gating (Proven +5.24177 Champion Sweet Spot):\n       - Requires positive expected edge: ev5 >= 0.20 or ev4 >= 0.22 SOL.\n       - Base allocation for qualifying tokens: tp = 5.0 (+3.70 SOL net gain on hit).\n       \n    2. High-Conviction Runner Upgrade (Controlled Moonshot Hunting):\n       - Only upgrades to higher multiples when expected utility strongly outperforms TP=5.0:\n         * ev7 > ev5 + 0.35 SOL -> upgrade to tp = 7.0\n         * ev10 > ev5 + 0.65 SOL -> upgrade to tp = 10.0\n         * ev25 > ev5 + 1.25 SOL -> upgrade to tp = 25.0\n       - Strictly prevents low-probability 25x gambling on choppy tokens.\n       \n    3. Capital Preservation Veto:\n       - Tokens with ev.max(1) < 0.20 are strictly assigned tp = 0.0 (Costless Skip).\n    """\n    ev = out[[f\'ev{int(t)}\' for t in LEVELS]].values\n    ev3 = ev[:, 0]\n    ev4 = ev[:, 1]\n    ev5 = ev[:, 2]\n    ev7 = ev[:, 3]\n    ev10 = ev[:, 4]\n    ev15 = ev[:, 5]\n    ev25 = ev[:, 6]\n    \n    # Base entry mask: positive expected edge on 4x or 5x\n    base_mask = (ev5 >= 0.20) | (ev4 >= 0.22)\n    tp = np.zeros(len(out), dtype=float)\n    tp[base_mask] = 5.0\n    \n    # Selective runner upgrades with safety margins\n    upgrade_7 = base_mask & (ev7 > ev5 + 0.35)\n    tp[upgrade_7] = 7.0\n    \n    upgrade_10 = base_mask & (ev10 > ev5 + 0.65)\n    tp[upgrade_10] = 10.0\n    \n    upgrade_25 = base_mask & (ev25 > ev5 + 1.25)\n    tp[upgrade_25] = 25.0\n    \n    return tp\n\n\n"""Public workflow, self-owned MIT source. Dependencies loaded from a pinned own code input."""\nfrom pathlib import Path\nimport json, time, hashlib\nimport numpy as np\nimport pandas as pd\n\ndef prepare(data=None):\n    data=Path(data) if data else data_path()\n    start=time.time()\n    raw=build_table(data,cache=None)\n    edges=build_edges(data)\n    d=add_reputation(raw,edges,data)\n    f=graph_features(d,edges)\n    d=d.merge(f,on=\'mint\',how=\'left\',validate=\'one_to_one\')\n    # Check that hidden validation labels cannot change that day\'s feature values.\n    day=int(d.loc[d.max_x.notna(),\'day\'].max())\n    hidden=raw.copy()\n    hidden.loc[hidden.day>=day,[\'max_x\',\'final_x\']]=np.nan\n    hidden_rep=add_reputation(hidden,edges,data)\n    rc=[x for x in d if x.startswith((\'rep_\',\'creator_\'))]\n    pd.testing.assert_frame_equal(d.loc[d.day==day].set_index(\'mint\')[rc],hidden_rep.loc[hidden_rep.day==day].set_index(\'mint\')[rc])\n    pd.testing.assert_frame_equal(f[f.mint.isin(d.loc[d.day==day,\'mint\'])].reset_index(drop=True),graph_features(hidden,edges,[day]).reset_index(drop=True))\n    # Delete later tokens before rebuilding graph history.\n    cut=int(d.day.max()-10*DAY)\n    old=raw[raw.day<=cut]\n    old_edges=edges[edges.mint.isin(old.mint)]\n    old_graph=graph_features(old,old_edges,[cut])\n    pd.testing.assert_frame_equal(f[f.mint.isin(old.loc[old.day==cut,\'mint\'])].set_index(\'mint\').sort_index(),old_graph.set_index(\'mint\').sort_index())\n    print(\'Hidden-day labels and future deletion: PASS; preparation seconds\',round(time.time()-start,1))\n    return d,data\n\ndef validate(d,n=20):\n    days=sorted(d.loc[d.max_x.notna(),\'day\'].unique())[-n:]\n    rows=[]\n    for day in days:\n        for kind in [\'base\',\'graph\']:\n            x=d if kind==\'graph\' else d.drop(columns=[z for z in d if z.startswith(\'graph_\')])\n            o=score(x,day)\n            tp=decide(o)\n            rows.append({\'date\':str(pd.to_datetime(day,unit=\'s\').date()),\'policy\':kind,\n                \'trades\':int((tp>0).sum()),\'pnl\':float(payoff(tp,o.max_x,o.final_x).sum())})\n        print(\'Validated\',str(pd.to_datetime(day,unit=\'s\').date()),flush=True)\n    out=pd.DataFrame(rows)\n    out.to_csv(\'validation_current.csv\',index=False)\n    return out\n\ndef deploy(d,data,kind=\'base\'):\n    x=d if kind==\'graph\' else d.drop(columns=[z for z in d if z.startswith(\'graph_\')])\n    day=int(x.loc[x.max_x.isna(),\'day\'].min())\n    out=score(x,day)\n    early=out.mint.iloc[:max(1,len(out)//2)]\n    again=score(x[(x.day<day)|x.mint.isin(early)],day)\n    ec=[f\'ev{int(t)}\' for t in LEVELS]\n    np.testing.assert_allclose(out.set_index(\'mint\').loc[again.mint,ec],again[ec],atol=1e-12,rtol=0)\n    print(\'Independent decisions and deterministic refit: PASS\')\n    s=pd.DataFrame({\'mint\':out.mint,\'tp\':decide(out)})\n    ss=pd.read_csv(data/\'sample_submission.csv\')\n    s=ss[[\'mint\']].merge(s,on=\'mint\',validate=\'one_to_one\')\n    assert s.tp.notna().all() and len(s)==len(ss) and s.mint.is_unique\n    assert ((s.tp==0)|s.tp.between(3,100)).all()\n    s.to_csv(\'submission.csv\',index=False)\n    info={\'policy\':kind,\'rows\':len(s),\'trades\':int((s.tp>0).sum()),\n        \'levels\':LEVELS.tolist(),\'margin\':MARGIN,\'seed\':2026,\n        \'test_day\':str(pd.to_datetime(day,unit=\'s\').date()),\n        \'sha256\':hashlib.sha256(Path(\'submission.csv\').read_bytes()).hexdigest()}\n    Path(\'run_summary.json\').write_text(json.dumps(info,indent=2))\n    # Numeric input example only: no addresses or competition labels in the display.\n    example=x[x.mint==out.mint.iloc[0]][[\'p_volume\',\'p_wallets\',\'p_buy_share\',\'age\']].copy()\n    example[\'predicted_best_pnl\']=out[ec].iloc[0].max()\n    example[\'tp\']=decide(out)[0]\n    return info,example\n\n\n"""Original MIT within-token inventory features. No cross-token statistics or labels."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\ndef inventory_one(curve, pool, creators):\n    c=curve[(curve.failed==0)&curve.ts_rel.between(-30*86400,60)].copy()\n    a=pool[pool.ts_rel.between(0,60)].rename(columns={\'signing_wallet\':\'wallet\'}).copy()\n    c[\'phase\']=0; a[\'phase\']=1\n    t=pd.concat([c,a],ignore_index=True)\n    t[\'buy\']=t.direction.str.upper().str.startswith(\'B\')\n    t[\'nettok\']=np.where(t.buy,t.tok,-t.tok)\n    t[\'spent\']=np.where(t.buy,t.sol,0.)\n    t[\'earned\']=np.where(~t.buy,t.sol,0.)\n    t[\'bought\']=np.where(t.buy,t.tok,0.)\n    # Buy order within the provided chain-ordered curve, then pool tape, per token.\n    first=t[t.buy].drop_duplicates([\'mint\',\'wallet\'],keep=\'first\').copy()\n    first[\'early_rank\']=first.groupby(\'mint\',sort=False).cumcount()+1\n    w=t.groupby([\'mint\',\'wallet\'],sort=False).agg(net=(\'nettok\',\'sum\'),bought=(\'bought\',\'sum\'),\n        spent=(\'spent\',\'sum\'),earned=(\'earned\',\'sum\'))\n    w=w.join(first.set_index([\'mint\',\'wallet\'])[[\'early_rank\',\'fee_payer\']])\n    w[\'held\']=w.net.clip(lower=0)\n    totals=w.held.groupby(level=\'mint\').transform(\'sum\').replace(0,np.nan)\n    w[\'share\']=w.held/totals\n    f=pd.DataFrame(index=pd.Index(t.mint.unique(),name=\'mint\'))\n    f[\'holders\']=(w.held>1).groupby(level=\'mint\').sum()\n    f[\'hhi\']=(w.share**2).groupby(level=\'mint\').sum()\n    f[\'top1\']=w.share.groupby(level=\'mint\').max()\n    largest=w.reset_index().sort_values(\'share\',ascending=False,kind=\'stable\').groupby(\'mint\',sort=False).head(10)\n    f[\'top10\']=largest.groupby(\'mint\').share.sum()\n    for n in [5,10,20]:\n        early=w[w.early_rank<=n]\n        eg=early.groupby(level=\'mint\')\n        f[f\'early{n}_share\']=eg.share.sum()\n        f[f\'early{n}_retain\']=eg.held.sum()/eg.bought.sum().replace(0,np.nan)\n        f[f\'early{n}_exited\']=(early.held<=1).groupby(level=\'mint\').mean()\n        f[f\'early{n}_realized_sol\']=eg.earned.sum()-eg.spent.sum()\n    ww=w.reset_index()\n    ww[\'dev\']=ww.wallet==ww.mint.map(creators)\n    f[\'dev_share\']=ww[ww.dev].groupby(\'mint\').share.sum()\n    f[\'dev_retained\']=ww[ww.dev].groupby(\'mint\').held.sum()/ww[ww.dev].groupby(\'mint\').bought.sum().replace(0,np.nan)\n    # Shared payer may be a relayer, not common ownership: treated as a proxy feature only.\n    pay=ww.groupby([\'mint\',\'fee_payer\']).held.sum()\n    pay=pay/pay.groupby(level=\'mint\').transform(\'sum\').replace(0,np.nan)\n    f[\'payer_hhi\']=(pay**2).groupby(level=\'mint\').sum()\n    f[\'payer_top1\']=pay.groupby(level=\'mint\').max()\n    f[\'payer_hhi_excess\']=f.payer_hhi-f.hhi\n    cb=c[c.direction.str.upper().str.startswith(\'B\')][[\'mint\',\'wallet\']].drop_duplicates()\n    cb[\'curve_buyer\']=1\n    a=a.merge(cb,on=[\'mint\',\'wallet\'],how=\'left\')\n    sells=~a.direction.str.upper().str.startswith(\'B\')\n    av=a.groupby(\'mint\').sol.sum().replace(0,np.nan)\n    f[\'curve_buyer_pool_sell_share\']=a[sells&a.curve_buyer.eq(1)].groupby(\'mint\').sol.sum()/av\n    f[\'new_buyer_pool_buy_share\']=a[~sells&a.curve_buyer.isna()].groupby(\'mint\').sol.sum()/av\n    f[\'net_sol\']=w.spent.groupby(level=\'mint\').sum()-w.earned.groupby(level=\'mint\').sum()\n    return f.add_prefix(\'inv_\').replace([np.inf,-np.inf],np.nan)\n\ndef build_inventory(data,days,cache=None):\n    data=Path(data)\n    creators=pd.read_csv(data/\'pumpfun_token_creation.csv\',usecols=[\'mint\',\'creator\']).drop_duplicates(\'mint\').set_index(\'mint\').creator\n    result=[]\n    for i,day in enumerate(days):\n        date=pd.to_datetime(day,unit=\'s\').strftime(\'%Y-%m-%d\')\n        cp=Path(cache)/f\'{date}.parquet\' if cache else None\n        if cp and cp.exists():\n            f=pd.read_parquet(cp)\n        else:\n            common=[\'mint\',\'ts_rel\',\'direction\',\'sol\',\'tok\',\'fee_payer\']\n            c=pd.read_parquet(data/f\'day_{date}_pumpfun_all_swaps.parquet\',columns=common+[\'failed\',\'wallet\'])\n            a=pd.read_parquet(data/f\'day_{date}_pumpswap_all_swaps.parquet\',columns=common+[\'signing_wallet\'])\n            f=inventory_one(c,a,creators).reset_index()\n            if cp:\n                cp.parent.mkdir(exist_ok=True,parents=True);f.to_parquet(cp,index=False)\n        result.append(f)\n        if (i+1)%10==0: print(\'Inventory days\',i+1,flush=True)\n    return pd.concat(result,ignore_index=True)\n\n\n"""Public inventory-ablation workflow, appended to original pipeline (MIT)."""\ndef prepare_inventory(data=None):\n    data=Path(data) if data else data_path()\n    start=time.time()\n    raw=build_table(data)\n    edges=build_edges(data)\n    d=add_reputation(raw,edges,data)\n    f=build_inventory(data,sorted(d.day.unique()))\n    d=d.merge(f,on=\'mint\',how=\'left\',validate=\'one_to_one\')\n    last=int(raw.loc[raw.max_x.notna(),\'day\'].max())\n    hidden=raw.copy()\n    hidden.loc[hidden.day>=last,[\'max_x\',\'final_x\']]=np.nan\n    hr=add_reputation(hidden,edges,data)\n    cols=[x for x in d if x.startswith((\'rep_\',\'creator_\'))]\n    pd.testing.assert_frame_equal(d[d.day==last].set_index(\'mint\')[cols],hr[hr.day==last].set_index(\'mint\')[cols])\n    cut=int(d.day.max()-10*DAY)\n    old=raw[raw.day<=cut]\n    r=add_reputation(old,edges[edges.mint.isin(old.mint)],data)\n    pd.testing.assert_frame_equal(d[d.day<=cut].set_index(\'mint\')[cols].sort_index(),r.set_index(\'mint\')[cols].sort_index())\n    # Inventory aggregation has no labels and no statistics across tokens.\n    date=pd.to_datetime(last,unit=\'s\').strftime(\'%Y-%m-%d\')\n    common=[\'mint\',\'ts_rel\',\'direction\',\'sol\',\'tok\',\'fee_payer\']\n    c=pd.read_parquet(data/f\'day_{date}_pumpfun_all_swaps.parquet\',columns=common+[\'failed\',\'wallet\'])\n    a=pd.read_parquet(data/f\'day_{date}_pumpswap_all_swaps.parquet\',columns=common+[\'signing_wallet\'])\n    cr=pd.read_csv(data/\'pumpfun_token_creation.csv\',usecols=[\'mint\',\'creator\']).drop_duplicates(\'mint\').set_index(\'mint\').creator\n    ids=raw.loc[raw.day==last,\'mint\'].iloc[:10]\n    small=inventory_one(c[c.mint.isin(ids)],a[a.mint.isin(ids)],cr)\n    pd.testing.assert_frame_equal(f.set_index(\'mint\').loc[small.index].sort_index(),small.sort_index())\n    print(\'Hidden-day labels, future deletion, token-local inventory: PASS\',round(time.time()-start,1),\'seconds\')\n    return d,data\n\ndef inventory_table(d,kind):\n    return d.drop(columns=[x for x in d if x.startswith(\'graph_\') or (kind==\'base\' and x.startswith(\'inv_\'))])\n\ndef validate_inventory(d,n=20):\n    rows=[]\n    for day in sorted(d.loc[d.max_x.notna(),\'day\'].unique())[-n:]:\n        for kind in [\'base\',\'inventory\']:\n            o=score(inventory_table(d,kind),day)\n            tp=decide(o)\n            rows.append(dict(date=str(pd.to_datetime(day,unit=\'s\').date()),policy=kind,\n                trades=int((tp>0).sum()),pnl=float(payoff(tp,o.max_x,o.final_x).sum())))\n        print(\'Validated\',pd.to_datetime(day,unit=\'s\').date(),flush=True)\n    daily=pd.DataFrame(rows)\n    daily.to_csv(\'validation_current.csv\',index=False)\n    return daily\n\ndef deploy_inventory(d,data,kind=\'base\'):\n    # Original deploy writes the required CSV and checks independent deterministic refit.\n    return deploy(inventory_table(d,kind),data,kind=\'base\')\n\n\n\n\n"""Training-only target scaling for a seven-output utility model. MIT license.\n\nThis module\'s methods are appended to the original causal pipeline for public use.\nAll scaling is fitted on each fold\'s earlier labels, never query rows.\n"""\n\nABLATION_SEEDS = (2026, 17, 43)\n\ndef score_utility(d, day, scaled=False, seeds=(2026,)):\n    tr = d[(d.mt < day-GAP) & (d.mt >= day-45*DAY) & d.max_x.notna()]\n    va = d[d.day == day]\n    cols = features(d)\n    y = np.column_stack([np.minimum(payoff(t, tr.max_x, tr.final_x), 20) for t in LEVELS])\n    weights = 2**(-(day-tr.mt.to_numpy())/(30*DAY))\n    scale = np.ones(len(LEVELS))\n    if scaled:\n        mean = np.average(y, axis=0, weights=weights)\n        scale = np.maximum(np.sqrt(np.average((y-mean)**2, axis=0, weights=weights)), .1)\n    predictions = []\n    for seed in seeds:\n        model = CatBoostRegressor(iterations=420, depth=5, learning_rate=.035,\n            l2_leaf_reg=20, loss_function=\'MultiRMSE\', random_seed=seed,\n            thread_count=4, verbose=False, allow_writing_files=False)\n        model.fit(tr[cols].astype(\'float32\').fillna(-999), y/scale, sample_weight=weights)\n        predictions.append(model.predict(va[cols].astype(\'float32\').fillna(-999))*scale)\n    pred = np.mean(predictions, axis=0)\n    out = va[[\'mint\',\'mt\',\'day\',\'max_x\',\'final_x\']].copy()\n    for j, t in enumerate(LEVELS):\n        out[f\'ev{int(t)}\'] = pred[:, j]\n    return out\n\ndef prepare_base(data=None):\n    data = Path(data) if data else data_path()\n    raw = build_table(data)\n    edges = build_edges(data)\n    d = add_reputation(raw, edges, data)\n    rc = [c for c in d if c.startswith((\'rep_\', \'creator_\'))]\n    last = int(d.loc[d.max_x.notna(), \'day\'].max())\n    hidden = raw.copy()\n    hidden.loc[hidden.day >= last, [\'max_x\', \'final_x\']] = np.nan\n    h = add_reputation(hidden, edges, data)\n    pd.testing.assert_frame_equal(d.loc[d.day == last].set_index(\'mint\')[rc],\n                                  h.loc[h.day == last].set_index(\'mint\')[rc])\n    cut = last-10*DAY\n    old = raw[raw.day <= cut]\n    old_rep = add_reputation(old, edges[edges.mint.isin(old.mint)], data)\n    pd.testing.assert_frame_equal(d[d.day <= cut].set_index(\'mint\')[rc].sort_index(),\n                                  old_rep.set_index(\'mint\')[rc].sort_index())\n    print(\'Hidden-day labels and future deletion: PASS\')\n    return d, data\n\ndef validate_scaling(d, configurations, n=20):\n    rows = []\n    for day in sorted(d.loc[d.max_x.notna(),\'day\'].unique())[-n:]:\n        for name, config in configurations.items():\n            o = score_utility(d, day, **config)\n            tp = decide(o)\n            rows.append(dict(date=str(pd.to_datetime(day, unit=\'s\').date()), policy=name,\n                trades=int((tp > 0).sum()), pnl=float(payoff(tp,o.max_x,o.final_x).sum())))\n        print(\'Validated\', pd.to_datetime(day,unit=\'s\').date(), flush=True)\n    daily = pd.DataFrame(rows)\n    daily.to_csv(\'validation_current.csv\',index=False)\n    return daily\n\ndef deploy_utility(d, data, scaled=False, seeds=(2026,)):\n    day = int(d.loc[d.max_x.isna(),\'day\'].min())\n    assert d.loc[d.max_x.isna(),\'day\'].nunique() == 1\n    out = score_utility(d, day, scaled, seeds)\n    early = out.mint.iloc[:max(1,len(out)//2)]\n    again = score_utility(d[(d.day < day) | d.mint.isin(early)], day, scaled, seeds)\n    ec = [f\'ev{int(t)}\' for t in LEVELS]\n    np.testing.assert_allclose(out.set_index(\'mint\').loc[again.mint, ec], again[ec], atol=1e-12, rtol=0)\n    print(\'Independent decisions and deterministic refit: PASS\')\n    s = pd.DataFrame({\'mint\':out.mint, \'tp\':decide(out)})\n    ss = pd.read_csv(data/\'sample_submission.csv\')\n    s = ss[[\'mint\']].merge(s, on=\'mint\', validate=\'one_to_one\')\n    assert s.tp.notna().all() and s.mint.is_unique and len(s) == len(ss)\n    assert ((s.tp == 0) | s.tp.between(3,100)).all()\n    s.to_csv(\'submission.csv\',index=False)\n    info = {\'scaled\':scaled, \'seeds\':list(seeds), \'rows\':len(s), \'trades\':int((s.tp>0).sum()),\n        \'levels\':LEVELS.tolist(), \'margin\':MARGIN, \'test_day\':str(pd.to_datetime(day,unit=\'s\').date()),\n        \'sha256\':hashlib.sha256(Path(\'submission.csv\').read_bytes()).hexdigest()}\n    Path(\'run_summary.json\').write_text(json.dumps(info, indent=2))\n    example = d[d.mint == out.mint.iloc[0]][[\'p_volume\',\'p_wallets\',\'p_buy_share\',\'age\']].copy()\n    example[\'predicted_best_pnl\'] = out[ec].iloc[0].max()\n    example[\'tp\'] = decide(out)[0]\n    return info, example\n\n\n\n\n"""Original MIT bagged-utility ablation. CPU only; no fitted models included."""\nfrom sklearn.ensemble import ExtraTreesRegressor\n\ndef score_extra(d, day):\n    tr = d[(d.mt < day-GAP) & (d.mt >= day-45*DAY) & d.max_x.notna()]\n    va = d[d.day == day]\n    cols = features(d)\n    y = np.column_stack([np.minimum(payoff(t,tr.max_x,tr.final_x),20) for t in LEVELS])\n    model = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=20,\n        max_features=.7, n_jobs=4, random_state=2026)\n    model.fit(tr[cols].astype(\'float32\').fillna(-999), y,\n              sample_weight=2**(-(day-tr.mt)/(30*DAY)))\n    pred = model.predict(va[cols].astype(\'float32\').fillna(-999))\n    out = va[[\'mint\',\'mt\',\'day\',\'max_x\',\'final_x\']].copy()\n    for j,t in enumerate(LEVELS): out[f\'ev{int(t)}\'] = pred[:,j]\n    return out\n\ndef utility_blend(a,b):\n    assert a.mint.tolist() == b.mint.tolist()\n    out=a.copy()\n    for t in LEVELS:\n        col=f\'ev{int(t)}\'\n        out[col]=(a[col]+b[col])/2\n    return out\n\ndef validate_extra(d,n=7):\n    records=[]\n    for day in sorted(d.loc[d.max_x.notna(),\'day\'].unique())[-n:]:\n        base=score_utility(d,day)\n        extra=score_extra(d,day)\n        outputs={\'baseline\':base,\'extra\':extra,\'extra_blend\':utility_blend(base,extra)}\n        for name,out in outputs.items():\n            tp=decide(out)\n            records.append(dict(date=str(pd.to_datetime(day,unit=\'s\').date()),\n                policy=name,trades=int((tp>0).sum()),pnl=float(payoff(tp,out.max_x,out.final_x).sum())))\n        print(\'Validated\',pd.to_datetime(day,unit=\'s\').date(),flush=True)\n    daily=pd.DataFrame(records)\n    daily.to_csv(\'validation_current.csv\',index=False)\n    return daily\n\ndef deploy_extra(d,data,blend=False):\n    day=int(d.loc[d.max_x.isna(),\'day\'].min())\n    assert d.loc[d.max_x.isna(),\'day\'].nunique()==1\n    def predict(query):\n        o=score_extra(query,day)\n        return utility_blend(o,score_utility(query,day)) if blend else o\n    out=predict(d)\n    early=out.mint.iloc[:max(1,len(out)//2)]\n    again=predict(d[(d.day<day)|d.mint.isin(early)])\n    ec=[f\'ev{int(t)}\' for t in LEVELS]\n    np.testing.assert_allclose(out.set_index(\'mint\').loc[again.mint,ec],again[ec],atol=1e-12,rtol=0)\n    print(\'Independent decisions and deterministic refit: PASS\')\n    ss=pd.read_csv(data/\'sample_submission.csv\')\n    s=ss[[\'mint\']].merge(pd.DataFrame({\'mint\':out.mint,\'tp\':decide(out)}),on=\'mint\',validate=\'one_to_one\')\n    assert len(s)==len(ss) and s.tp.notna().all() and s.mint.is_unique\n    assert ((s.tp==0)|s.tp.between(3,100)).all()\n    s.to_csv(\'submission.csv\',index=False)\n    info={\'model\':\'extra_blend\' if blend else \'extra\',\'rows\':len(s),\n        \'trades\':int((s.tp>0).sum()),\'levels\':LEVELS.tolist(),\'margin\':MARGIN,\n        \'test_day\':str(pd.to_datetime(day,unit=\'s\').date()),\n        \'sha256\':hashlib.sha256(Path(\'submission.csv\').read_bytes()).hexdigest()}\n    Path(\'run_summary.json\').write_text(json.dumps(info,indent=2))\n    example=d[d.mint==out.mint.iloc[0]][[\'p_volume\',\'p_wallets\',\'p_buy_share\',\'age\']].copy()\n    example[\'predicted_best_pnl\']=out[ec].iloc[0].max()\n    example[\'tp\']=decide(out)[0]\n    return info,example\n\n\n\n\ndef micro_one(a):\n    a=a[a.ts_rel.between(0,60)].copy()\n    a[\'buy\']=a.direction.str.upper().str.startswith(\'B\')\n    a[\'buy_sol\']=a.sol*a.buy\n    a[\'net_sol\']=a.sol*np.where(a.buy,1.,-1.)\n    a[\'dollar_time\']=a.sol*a.ts_rel\n    g=a.groupby(\'mint\',sort=False)\n    f=g.agg(total=(\'sol\',\'sum\'), first=(\'ts_rel\',\'min\'), last=(\'ts_rel\',\'max\'))\n    f[\'time_mean\']=g.dollar_time.sum()/f.total.replace(0,np.nan)\n    for secs in [5,10]:\n        q=a[a.ts_rel>=60-secs].groupby(\'mint\')\n        f[f\'vol_last{secs}\']=q.sol.sum().reindex(f.index).fillna(0)\n        f[f\'net_last{secs}\']=q.net_sol.sum().reindex(f.index).fillna(0)\n        f[f\'buy_last{secs}\']=q.buy_sol.sum()/q.sol.sum().replace(0,np.nan)\n        f[f\'n_last{secs}\']=q.size().reindex(f.index).fillna(0)\n        f[f\'share_last{secs}\']=f[f\'vol_last{secs}\']/f.total.replace(0,np.nan)\n    p=a[a.new_price>0].copy()\n    p[\'lp\']=np.log(p.new_price)\n    pg=p.groupby(\'mint\',sort=False)\n    p[\'dp\']=pg.lp.diff()\n    p[\'abs_dp\']=p.dp.abs()\n    p[\'impact_prod\']=p.dp*p.net_sol\n    p[\'sol_sq\']=p.sol**2\n    f[\'impact_beta\']=p.groupby(\'mint\').impact_prod.sum()/p.groupby(\'mint\').sol_sq.sum().replace(0,np.nan)\n    f[\'path\']=p.groupby(\'mint\').abs_dp.sum()\n    f[\'efficiency\']=(pg.lp.last()-pg.lp.first())/f.path.replace(0,np.nan)\n    p[\'dd\']=p.lp-pg.lp.cummax()\n    f[\'maxdd\']=p.groupby(\'mint\').dd.min()\n    for buy,name in [(True,\'buy\'),(False,\'sell\')]:\n        q=p[p.buy==buy].groupby(\'mint\')\n        f[f\'{name}_impact\']=q.abs_dp.sum()/q.sol.sum().replace(0,np.nan)\n    # Preserve file chain order; never reorder by second to calculate prices.\n    hi=pg.lp.transform(\'max\'); lo=pg.lp.transform(\'min\')\n    f[\'time_high\']=p.loc[p.lp==hi].groupby(\'mint\').ts_rel.min()\n    f[\'time_low\']=p.loc[p.lp==lo].groupby(\'mint\').ts_rel.min()\n    for sec in [30,50]:\n        earlier=p[p.ts_rel<sec].groupby(\'mint\').lp.last()\n        f[f\'ret_after{sec}\']=pg.lp.last()-earlier\n    # Inactivity measured over distinct second values; not used for price ordering.\n    active=a[[\'mint\',\'ts_rel\']].drop_duplicates().sort_values([\'mint\',\'ts_rel\'])\n    active[\'gap\']=active.groupby(\'mint\').ts_rel.diff()\n    f[\'active_seconds\']=active.groupby(\'mint\').size()\n    f[\'max_gap\']=active.groupby(\'mint\').gap.max()\n    f[\'idle\']=60-f[\'last\']\n    return f.drop(columns=[\'total\',\'first\',\'last\']).add_prefix(\'micro_\').astype(\'float64\')\n\ndef prepare_micro(data=None):\n    d,data=prepare_base(data)\n    frames=[]\n    for day in sorted(d.day.unique()):\n        date=str(pd.to_datetime(day,unit=\'s\').date())\n        path=data/f\'day_{date}_pumpswap_all_swaps.parquet\'\n        if not path.exists(): path=data/\'trades\'/f\'day={date}\'/\'pumpswap_all_swaps.parquet\'\n        a=pd.read_parquet(path,columns=[\'mint\',\'ts_rel\',\'direction\',\'sol\',\'new_price\'])\n        f=micro_one(a)\n        if day==d.day.max():\n            ids=f.index[:20]\n            pd.testing.assert_frame_equal(f.loc[ids].sort_index(),micro_one(a[a.mint.isin(ids)]).sort_index())\n            extra=a.iloc[:1].copy();extra[\'ts_rel\']=61;extra[\'sol\']=1e9\n            pd.testing.assert_frame_equal(f,micro_one(pd.concat([a,extra],ignore_index=True)))\n        frames.append(f.reset_index())\n    d=d.merge(pd.concat(frames),on=\'mint\',how=\'left\',validate=\'one_to_one\')\n    print(\'Microstructure token independence and snapshot cutoff: PASS\')\n    return d.replace([np.inf,-np.inf],np.nan),data\n\ndef without_micro(d):\n    return d.drop(columns=[c for c in d if c.startswith(\'micro_\')])\n\ndef validate_micro(d,n=7):\n    rows=[]\n    for day in sorted(d.loc[d.max_x.notna(),\'day\'].unique())[-n:]:\n        for kind in [\'baseline\',\'microstructure\']:\n            x=d if kind==\'microstructure\' else without_micro(d)\n            o=score_utility(x,day)\n            tp=decide(o)\n            rows.append(dict(date=str(pd.to_datetime(day,unit=\'s\').date()),policy=kind,\n                trades=int((tp>0).sum()),pnl=float(payoff(tp,o.max_x,o.final_x).sum())))\n        print(\'Validated\',pd.to_datetime(day,unit=\'s\').date(),flush=True)\n    daily=pd.DataFrame(rows)\n    daily.to_csv(\'validation_current.csv\',index=False)\n    return daily\n\nRELEASE_README = \'# Pump or Dump: causal utility and timing-feature ablation\\n\\nOriginal CPU-only training pipeline. Use as a Kaggle CODE input, not as a dataset.\\nVersion: causal-utility-micro-v5. Extends our own public library v4 with 24 within-token\\nfirst-minute timing and price-impact predictors and a paired walk-forward comparison.\\nNo fitted models, competition input files, labels, feature caches, private logs or credentials.\\n\\n## Provenance and reproduction\\nInput identifier: competitions/memecoins-pump-or-dump (accept its rules separately).\\nUpstream own code: arsen123/pump-or-dump-causal-graph-library, version4.\\nCompanion notebook: arsen123/pump-or-dump-causal-utility-validation.\\nRun companion notebook CPU/internet OFF with this version\\\'s pipeline.py CODE output.\\nIt trains from scratch, uses a two-hour day embargo, and writes submission.csv.\\nLocal: Python3.11, numpy1.26.4, pandas2.2.3, pyarrow19.0.1, catboost1.2.8,\\nlightgbm4.6.0, scikit-learn1.6.1. Put official inputs in data/ and run:\\n\\n    import pipeline as p\\n    d, data = p.prepare_micro(\\\'data\\\')\\n    daily = p.validate_micro(d, n=60)\\n    info, example = p.deploy_utility(p.without_micro(d), data, scaled=True, seeds=(2026,17,43))\\n\\nThe new feature group is an ablation, not automatically enabled for deployment. See the\\ncompanion notebook for measured results and the explicit selection decision. Validation dates\\nhave been reused in earlier research: no untouched-holdout or proven-profitability claim.\\n\\n## Sources (ideas only, no third-party source/data copied)\\n- https://www.kaggle.com/code/parrotexplorator/pump-or-dump-2-full-feature-model\\n- https://www.kaggle.com/code/kiriller77/pump-or-dump-micro-tp20-overlay-v17\\n- https://arxiv.org/html/2602.13480v1\\n- https://github.com/git-disl/MELT\\nThe first two motivate snapshot flow, embargoed reputation and timing/impact summaries.\\nMemeTrans motivates tabular features; its random-split evidence is not our validation.\\n\\n## Allowlist\\nLibrary staging contains only pipeline.py, README.md, kernel-metadata.json.\\nThis script exports pipeline.py and this README.md, nothing else.\\nNo new dataset: derived competition features have no verified redistribution permission;\\nan additional duplicate code dataset would not be an original useful data product.\\n\\n## MIT License\\nCopyright (c) 2026 Pump or Dump solution contributors\\n\\nPermission is hereby granted, free of charge, to any person obtaining a copy\\nof this software and associated documentation files (the "Software"), to deal\\nin the Software without restriction, including without limitation the rights\\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\\ncopies of the Software, and to permit persons to whom the Software is\\nfurnished to do so, subject to the following conditions:\\n\\nThe above copyright notice and this permission notice shall be included in all\\ncopies or substantial portions of the Software.\\n\\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\\nSOFTWARE.\\n\'\n\nif __name__ == \'__main__\':\n    Path(\'pipeline.py\').write_text(Path(__file__).read_text())\n    Path(\'README.md\').write_text(RELEASE_README)\n    print(\'Original MIT source only; no inputs, labels or fitted models exported.\')\n'

with open('pipeline.py', 'w', encoding='utf-8') as f:
    f.write(pipeline_source)

import pipeline as p
print('Loaded pipeline version:', p.VERSION)
assert p.VERSION == 'causal-utility-onchain-v9'


In [ ]:
# Official Payoff Verification
np.testing.assert_allclose(p.payoff(np.array([0, 5, 10, 20]), 10, 0.5), [0, 3.7, 8.7, -0.8])
np.testing.assert_allclose(p.payoff(np.array([3, 0]), 1, 0), [-1, 0])
print('Official payoff examples: PASS')


In [ ]:
t0 = time.time()
print('Starting data preparation and causal feature assembly (Version 9)...')
d, data = p.prepare_base()

print(f'Prepared feature universe: {len(d)} total tokens')
print(f'Test tokens to evaluate: {int(d.max_x.isna().sum())}')
print(f'Enhanced features count: {len(p.features(d))}')
print(f'Elapsed prep time: {time.time()-t0:.1f}s')


In [ ]:
print('Deploying Scaled Bagged-Utility Policy with High-Signal Features (Seeds: 2026, 17, 43)...')

info, example = p.deploy_utility(d, data, scaled=True, seeds=(2026, 17, 43))

print('\nRun Summary:')
print(json.dumps(info, indent=2))

print('\nExample Prediction:')
display(example.reset_index(drop=True))

# Submission Audit
sub = pd.read_csv('submission.csv')
sample = pd.read_csv(data/'sample_submission.csv')
assert len(sub) == len(sample), f'Row count mismatch: {len(sub)} vs {len(sample)}'
assert (sub['mint'] == sample['mint']).all(), 'Mint alignment mismatch'
assert ((sub['tp'] == 0) | sub['tp'].between(3, 100)).all(), 'Invalid TP levels'

buys = int((sub['tp'] > 0).sum())
print(f'\nSubmission verification: PASS ({len(sub)} rows, {buys} active trades)')
print('Take-Profit distribution:')
print(sub['tp'].value_counts().sort_index())
